In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import scipy.stats as stats
import openpyxl

In [ ]:
# load necessary files
import sys
import os
import json
import tkinter as tk
from tkinter import filedialog

# Initialize tkinter (Standard Python GUI library, simpler than Qt)
root = tk.Tk()
root.withdraw()  # Hide the main window
root.wm_attributes('-topmost', 1)  # Bring the dialog to the front

# Center the hidden window on the screen so dialogs open in the middle
root.update_idletasks()
screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
x = (screen_width // 2) - 1
y = (screen_height // 2) - 1
root.geometry(f"1x1+{x}+{y}")

# Get absolute path to analysis directory to show dataset folders
# current_dir = os.getcwd()
DATA_SET = "/home/rcd/Desktop/Workspace/remote-control/analysis/2026_03_02/HD_Scenario_01_REAL/20260302_155501_Test_01"

print(f"File picker will start in: {DATA_SET}")

def select_file(title, initial_dir):
    root.update()

    file_path = filedialog.askopenfilename(
        parent=root, # Explicitly parent to the root window positioned at center
        title=title,
        initialdir=initial_dir,
        filetypes=[("JSON Files", "*.json"), ("All Files", "*.*")]
    )

    if not file_path:
        print(f"No file selected for '{title}'. Exiting.")
        root.destroy()
        raise SystemExit("File selection cancelled")

    print(f"Selected: {file_path}")
    return file_path

try:
    # 1. Select Hardware Usage JSON file
    hardware_usage_file = select_file("Select Hardware Usage JSON File", DATA_SET)

    # Extract directory from the first selected file to use as default for next selections
    default_dir = os.path.dirname(hardware_usage_file)

    # 2. Select Command Latency JSON file
    command_latency_file = select_file("Select Command Latency JSON File", default_dir)

    # 3. Select Train Telemetry/Video Latency JSON file
    video_telemetry_latency_file = select_file("Select Train Telemetry/Video Latency JSON File", default_dir)

    # 4. Select Telemetry JSON file
    telemetry_file = select_file("Select Telemetry JSON File", default_dir)
finally:
    # Cleanup the hidden window
    root.destroy()

print("\nLoading data...")

# Load the train telemetry/video latency data
with open(video_telemetry_latency_file, 'r') as file:
    data = json.load(file)

# load the telemetry data
with open(telemetry_file, 'r') as file:
    telemetry_data = json.load(file)

print(f"Data loaded successfully from {video_telemetry_latency_file}")


# Convert to DataFrames for easier analysis
telemetry_df = pd.DataFrame(data['telemetryLatencies'])
video_df = pd.DataFrame(data['videoLatencies'])
telemetry_data_df = pd.DataFrame(telemetry_data['data'])

# Normalize frameId to start from 1
if not video_df.empty and 'frameId' in video_df.columns:
    first_frame_id = video_df['frameId'].iloc[0]
    video_df['frameId'] = video_df['frameId'] - (first_frame_id - 1)
    print(f"\nFrameId normalized: First frame was {first_frame_id}, now starts from 1")
    print(f"New frameId range: {video_df['frameId'].min()} to {video_df['frameId'].max()}")

# For each video frame, find the telemetry entry with the closest timestamp
speeds = []
for _, video_row in video_df.iterrows():
    video_timestamp = video_row['received_at']

    # Calculate time difference between video frame and all telemetry entries
    time_diffs = abs(telemetry_data_df['timestamp'] - video_timestamp)

    # Find the index of the closest telemetry entry
    closest_idx = time_diffs.idxmin()

    # Get the speed from the closest telemetry entry's nested 'data' field
    telemetry_entry = telemetry_data_df.loc[closest_idx]
    if isinstance(telemetry_entry['data'], dict) and 'speed' in telemetry_entry['data']:
        speed = telemetry_entry['data']['speed']
    else:
        speed = np.nan

    speeds.append(speed)

# Add the speeds to video_df
video_df['speed'] = speeds

# export video_df to excel file
video_df.to_excel('video_latencies.xlsx', index=False)

# Create a summary DataFrame from statistics
protocols = ['websocket', 'webtransport', 'mqtt']
stats_data = []
for protocol in protocols:
    stats_data.append({
        'protocol': protocol,
        'count': data['statistics'][protocol]['count'],
        'avg': data['statistics'][protocol]['avg'],
        'min': data['statistics'][protocol]['min'],
        'max': data['statistics'][protocol]['max']
    })
stats_df = pd.DataFrame(stats_data)